# 04 · Model training, threshold tuning and validation

- **Model:** LightGBM binary classifier (MIT licence) on the tabular similarity features.
- **Leak-free validation — entities, not pairs, are held out.** Roles are assigned per S1
  entity (`P.entity_roles`): *train* (fit), *valid-A* (early stopping + threshold tuning) and
  *valid-B* (never touched until the final report). All candidate pairs of an entity share its
  role; each S2/S3 record has at most one true S1, so positives never straddle roles.
- **Metric = leaderboard metric:** macro F0.5 per S1 entity over *every* entity of the role
  (including singletons and entities whose true matches the candidate stage missed — they count
  against recall), using the same decision rule as inference.
- **Threshold** tuned on valid-A to maximise macro F0.5 (not 0.5), then reported on valid-B.
- **Memory:** features are streamed from the part files straight into per-role float32
  matrices (`P.load_training_matrices`); the train matrix is freed once LightGBM has binned it.

In [ ]:
import sys, time, gc
from pathlib import Path

# notebooks/ -> ../src holds the shared, importable pipeline package `ber`
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ber import config as C, io, pipeline as P

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
C.ensure_dirs()
print(C.describe())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB | CPUs: {psutil.cpu_count()}")
except ImportError:
    pass

In [ ]:
import lightgbm as lgb
from ber import model as M, metrics as MT
mats = P.load_training_matrices("train", roles=(1, 2))   # valid-B is loaded after training (peak RAM)
FEATS = mats["features"]
keys, n1 = P.true_pair_keys("train")
n_true_all = MT.true_counts(keys, n1)
role = P.entity_roles(P.entity_ids("train")[0])
ents = {2: np.flatnonzero(role == 2), 3: np.flatnonzero(role == 3)}
print(f"entities: train {int((role == 1).sum()):,} | valid-A {len(ents[2]):,} | valid-B {len(ents[3]):,}")
print(f"positives: train {int(mats[1]['y'].sum()):,} | valid-A {int(mats[2]['y'].sum()):,}")

## 1. Train (early stopping on valid-A)

In [ ]:
t0 = time.time()
model = M.train_matrices(mats, C.LGB_PARAMS, C.LGB_NUM_BOOST_ROUND, C.LGB_EARLY_STOPPING, C.N_JOBS)
mats[1] = None                                  # free the training rows before loading valid-B
mats[2]["prob"] = M.predict_matrix(model, mats[2]["X"], C.N_JOBS)
mats[2]["X"] = None
gc.collect()
mats[3] = P.load_training_matrices("train", roles=(3,))[3]
mats[3]["prob"] = M.predict_matrix(model, mats[3]["X"], C.N_JOBS)
mats[3]["X"] = None
gc.collect()
print(f"trained + scored validation pairs in {time.time() - t0:.0f}s")

## 2. Precision-recall curve and macro F0.5 vs threshold (valid-A)

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
A = mats[2]
sw = MT.sweep_idx(ents[2], n_true_all, A["s1_idx"], A["y"], A["prob"])
best = sw.loc[sw.macro_f05.idxmax()]
THRESHOLD = float(best.threshold)
prec, rec, _ = precision_recall_curve(A["y"], A["prob"])
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(rec, prec); ax[0].set_xlim(0.8, 1.001); ax[0].set_ylim(0.8, 1.001); ax[0].grid(alpha=.3)
ax[0].set_xlabel("recall (candidate pairs)"); ax[0].set_ylabel("precision")
ax[0].set_title(f"valid-A pair-level PR curve (AP={average_precision_score(A['y'], A['prob']):.4f})")
ax[1].plot(sw.threshold, sw.macro_f05, label="macro F0.5")
ax[1].plot(sw.threshold, sw.pair_precision, ":", label="precision")
ax[1].plot(sw.threshold, sw.pair_recall, ":", label="recall")
ax[1].axvline(THRESHOLD, ls="--", c="k"); ax[1].axhline(C.TARGET_F05, ls="-.", c="g", label=f"target {C.TARGET_F05}")
ax[1].set_ylim(0.9, 1.0); ax[1].set_xlabel("threshold"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("valid-A: leaderboard metric vs threshold")
plt.tight_layout(); plt.show()
print(f"chosen threshold {THRESHOLD:.2f}: valid-A macro F0.5 {best.macro_f05:.5f} | P {best.pair_precision:.5f} | "
      f"R {best.pair_recall:.5f} | singleton acc {best.singleton_acc:.4f}")

## 3. Final, unbiased report on valid-B (threshold fixed from valid-A)

In [ ]:
B = mats[3]
rep = MT.sweep_idx(ents[3], n_true_all, B["s1_idx"], B["y"], B["prob"], thresholds=[THRESHOLD]).iloc[0]
ceiling = B["y"].sum() / max(1, n_true_all[ents[3]].sum())
print(f"VALID-B  macro F0.5 = {rep.macro_f05:.5f} | precision = {rep.pair_precision:.5f} | recall = {rep.pair_recall:.5f} | "
      f"singleton acc = {rep.singleton_acc:.4f} | threshold = {THRESHOLD:.2f} | entities = {len(ents[3]):,}")
print(f"valid-B candidate recall ceiling: {ceiling:.5f}")
print("TARGET MET" if rep.macro_f05 >= C.TARGET_F05 else "TARGET NOT MET - see next steps below")

## 4. Error analysis (valid-A + valid-B)

In [ ]:
cols = ["entity_id", "name_core", "addr_clean"]
s1c = pd.read_parquet(io.fetch("clean/train_s1.parquet"), columns=cols)
rc = pd.concat([pd.read_parquet(io.fetch(f"clean/train_{k}.parquet"), columns=cols) for k in ("s2", "s3")], ignore_index=True)
val = pd.DataFrame({k: np.concatenate([mats[2][k], mats[3][k]]) for k in ("s1_idx", "r_idx", "y", "prob")})
def pair_str(i1, ir, p):
    a, b = s1c.iloc[i1], rc.iloc[ir]
    return f"  S1 {a.name_core!r:36} | {a.addr_clean!r}\n  {b.entity_id[:2]} {b.name_core!r:36} | {b.addr_clean!r}   p={p:.3f}"
fp = val[(val.prob >= THRESHOLD) & (val.y == 0)].nlargest(10, "prob")
fn_all = val[(val.prob < THRESHOLD) & (val.y == 1)]
fn = fn_all.sample(min(10, len(fn_all)), random_state=0)
print("=== highest-confidence FALSE POSITIVES ===")
for _, x in fp.iterrows(): print(pair_str(int(x.s1_idx), int(x.r_idx), x.prob) + "\n")
print("=== sample FALSE NEGATIVES (in candidates, below threshold) ===")
for _, x in fn.iterrows(): print(pair_str(int(x.s1_idx), int(x.r_idx), x.prob) + "\n")
del s1c, rc, val; gc.collect()

In [ ]:
imp = M.feature_importance(model)
imp.head(30).plot.barh(x="feature", y="gain", figsize=(8, 9), legend=False, title="LightGBM gain").invert_yaxis()
plt.tight_layout(); plt.show()

## 5. If the target is not met - concrete next steps

In [ ]:
if rep.macro_f05 < C.TARGET_F05:
    tips = []
    if ceiling < 0.99:
        tips.append("Raise candidate recall: increase TOPK (see recall-vs-K in 02) or add retrieval parts / weights.")
    if rep.pair_precision < 0.99 or rep.singleton_acc < 0.97:
        tips.append("Precision / singleton errors dominate: add cluster-consistency features (similarity of a candidate to the "
                    "other high-confidence candidates of the same S1) and a second-stage model on first-stage scores.")
    if rep.pair_recall < 0.97:
        tips.append("Recall errors dominate: inspect FNs; better transliteration for Indic names, more abbreviation rules.")
    tips.append("More training entities (raise BER_TRAIN_PCT) and LightGBM tuning (num_leaves, learning rate, rounds).")
    print("\n".join(f"- {t}" for t in tips))
else:
    print(f"Target {C.TARGET_F05} met on the untouched valid-B entities.")

## 6. Save the model artifacts

The model trained on the *train* entities (early-stopped iteration count) is exactly the one
validated above and is what notebook 05 uses. Saved locally and to S3.

In [ ]:
mdir = C.CHECKPOINT_DIR / "model"; mdir.mkdir(parents=True, exist_ok=True)
model.save_model(str(mdir / "lgbm.txt"), num_iteration=model.best_iteration)
M.save_meta(mdir / "meta.json", threshold=THRESHOLD, one_to_one=C.ONE_TO_ONE, best_iteration=model.best_iteration,
            features=model.feature_name(), params=C.LGB_PARAMS, validA_macro_f05=float(best.macro_f05),
            validB_macro_f05=float(rep.macro_f05), validB_precision=float(rep.pair_precision),
            validB_recall=float(rep.pair_recall), validB_candidate_ceiling=float(ceiling),
            train_valid_pct=C.train_valid_pct(), sample_frac=C.SAMPLE_FRAC)
io.save_file(mdir / "lgbm.txt", force=True); io.save_file(mdir / "meta.json", force=True)
print("saved model/lgbm.txt + model/meta.json (local + S3)")